# Обучение и сохранение моделей
Датасет цен на дома `kc_house_data_cleaned.csv`

Целевая переменная `price`

In [29]:
import pandas as pd
import numpy as np
from math import sqrt
import pickle
import os
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
import optuna
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import GradientBoostingRegressor, BaggingRegressor, StackingRegressor, RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import LinearSVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
import tensorflow as tf
import keras_tuner as kt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam, SGD, RMSprop
from tensorflow.keras.callbacks import EarlyStopping
tf.get_logger().setLevel("ERROR")

optuna.logging.set_verbosity(optuna.logging.WARNING)

os.makedirs('./models', exist_ok=True)

df = pd.read_csv('./data/kc_house_data_cleaned.csv')
print(df.shape)
df.head()

(21613, 24)


,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,...,lat,long,sqft_living15,sqft_lot15,day,month,year,age,was_renovated,years_since_renovation
0,221900,3,1.00,1180,5650,1.0,0,0,3,7,...,47.5112,-122.257,1340,5650,13,10,2014,59,0,0
1,538000,3,2.25,2570,7242,2.0,0,0,3,7,...,47.7210,-122.319,1690,7639,9,12,2014,63,1,23
2,180000,2,1.00,770,10000,1.0,0,0,3,6,...,47.7379,-122.233,2720,8062,25,2,2015,82,0,0
3,604000,4,3.00,1960,5000,1.0,0,0,5,7,...,47.5208,-122.393,1360,5000,9,12,2014,49,0,0
4,510000,3,2.00,1680,8080,1.0,0,0,3,8,...,47.6168,-122.045,1800,7503,18,2,2015,28,0,0


In [ ]:
def print_metrics(y_true, y_pred):
    print(f'MAE: {mean_absolute_error(y_true, y_pred)}')
    print(f'MSE: {mean_squared_error(y_true, y_pred)}')
    print(f'RMSE: {sqrt(mean_squared_error(y_true, y_pred))}')
    print(f'MAPE: {mean_absolute_percentage_error(y_true, y_pred)}')
    print(f'R^2: {r2_score(y_true, y_pred)}')

X = df.drop(columns=['price'])
y = df['price']

# Feature engineering
X['sqft_ratio']      = X['sqft_living'] / X['sqft_lot'].clip(lower=1)
X['total_rooms']     = X['bedrooms'] + X['bathrooms']
X['rooms_per_floor'] = X['bedrooms'] / X['floors'].clip(lower=1)
X['renovated_age']   = X['age'] - X['years_since_renovation']
X['living_vs_15']    = X['sqft_living'] / X['sqft_living15'].clip(lower=1)

X = X.fillna(0)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_sc = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_sc  = pd.DataFrame(scaler.transform(X_test),  columns=X_test.columns)

with open('./models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

results = {}
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (17290, 28), Test: (4323, 28)


## 1. Полиномиальная регрессия

In [31]:
def objective_poly(trial):
    degree = trial.suggest_int('degree', 2, 3)
    
    pipeline = Pipeline([
        ('poly', PolynomialFeatures(degree=degree)),
        ('scaler', scaler), 
        ('linear', LinearRegression())               
    ])
    pipeline.fit(X_train, y_train)
    
    y_pred = pipeline.predict(X_test)  
    
    mse = mean_squared_error(y_test, y_pred)
    return mse

study_poly = optuna.create_study(direction='minimize')
study_poly.optimize(objective_poly, n_trials=10, show_progress_bar=True)

best_params = study_poly.best_params

best_pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=best_params['degree'])),
    ('scaler', scaler),
    ('linear', LinearRegression())
])

best_pipeline.fit(X_train, y_train)
y_pred_poly = best_pipeline.predict(X_test)

print_metrics(y_test, y_pred_poly)
with open("./models/polynomial.pkl", "wb") as f:
    pickle.dump(best_pipeline, f)


  0%|          | 0/10 [00:00<?, ?it/s]

MAE: 102415.97867780937
MSE: 29254583948.671047
RMSE: 171039.71453633523
MAPE: 0.20148789550353324
R^2: 0.8064874972345867


## 2. Градиентный бустинг

In [32]:
def objective_gb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 20),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'max_features': trial.suggest_float('max_features', 0.5, 1.0),
    }
    
    model = GradientBoostingRegressor(**params, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
    return scores.mean()

study_gb = optuna.create_study(direction='maximize')
study_gb.optimize(objective_gb, n_trials=20, show_progress_bar=True)

gb_model = GradientBoostingRegressor(**study_gb.best_params, random_state=42)
gb_model.fit(X_train, y_train)

y_pred_test_gb = gb_model.predict(X_test)
print_metrics(y_test, y_pred_test_gb)
with open("./models/gradient_boosting.pkl", "wb") as f:
    pickle.dump(gb_model, f)


  0%|          | 0/20 [00:00<?, ?it/s]

MAE: 71657.3694356065
MSE: 21658800961.592968
RMSE: 147169.2935418016
MAPE: 0.1296634657345817
R^2: 0.856731895817435


## 3. CatBoost

In [33]:
def objective_cat(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 100, 250),
        'depth': trial.suggest_int('depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
    }
    
    model = CatBoostRegressor(**params, random_seed=42, verbose=False)
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
    return scores.mean()

study_cat = optuna.create_study(direction='maximize')
study_cat.optimize(objective_cat, n_trials=20, show_progress_bar=False)

cat_model = CatBoostRegressor(**study_cat.best_params, random_seed=42, verbose=False)
cat_model.fit(X_train, y_train)

y_pred_test_cat = cat_model.predict(X_test)
print_metrics(y_test, y_pred_test_cat)
cat_model.save_model("./models/catboost.cbm")


MAE: 66858.08947429032
MSE: 14926679109.587248
RMSE: 122174.78917349213
MAPE: 0.12357455942402687
R^2: 0.901263369954586


## 4. Бэггинг

In [34]:
def objective_bagging(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 150),
        'max_samples': trial.suggest_float('max_samples', 0.5, 1.0),
    }
    
    base = DecisionTreeRegressor(max_depth=trial.suggest_int('max_depth', 5, 15),random_state=42)
    
    model = BaggingRegressor(estimator=base, **params, random_state=42)
    return cross_val_score(model, X_train, y_train, cv=5, scoring='r2').mean()

study_bag = optuna.create_study(direction='maximize')
study_bag.optimize(objective_bagging, n_trials=15, show_progress_bar=True)

base = DecisionTreeRegressor(max_depth=study_bag.best_params['max_depth'], random_state=42)
bagging_params = {k: v for k, v in study_bag.best_params.items() if k != 'max_depth'}

model = BaggingRegressor(estimator=base, **bagging_params, random_state=42)
model.fit(X_train, y_train)

y_pred_test_bag = model.predict(X_test)
print_metrics(y_test, y_pred_test_bag)
with open("./models/bagging.pkl", "wb") as f:
    pickle.dump(model, f)


  0%|          | 0/15 [00:00<?, ?it/s]

MAE: 75179.44533833441
MSE: 23234139205.982502
RMSE: 152427.48835424174
MAPE: 0.1360499300432717
R^2: 0.8463113871235235


## 5. Стэкинг

In [35]:
base_models = [
    ('rf', RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)),
    ('gb', GradientBoostingRegressor(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42)),
    ('dt', DecisionTreeRegressor(max_depth=10, random_state=42)),
]

stacking = StackingRegressor(
    estimators=base_models,
    final_estimator=Ridge(alpha=1.0),  
    cv=5,
    n_jobs=-1
)
stacking.fit(X_train, y_train)
print_metrics(y_test, stacking.predict(X_test))
with open("./models/stacking.pkl", "wb") as f:
    pickle.dump(stacking, f)


MAE: 76320.25507439257
MSE: 21708468006.33635
RMSE: 147337.93810942365
MAPE: 0.13962224170487306
R^2: 0.8564033594707854


## 6. Глубокая полносвязная нейронная сеть.

In [41]:
import tensorflow as tf
tf.get_logger().setLevel("ERROR")

N_FEATURES = X_train_sc.shape[1]
layers_map = {"64_32": (64, 32), "128_64": (128, 64), "64_32_16": (64, 32, 16)}

y_train_log = np.log1p(y_train)
y_test_log  = np.log1p(y_test)

def build_keras_rmsprop(lr, layers, dropout):
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Dense(layers[0], activation="relu", input_shape=(N_FEATURES,)))
    model.add(tf.keras.layers.BatchNormalization())
    for units in layers[1:]:
        model.add(tf.keras.layers.Dense(units, activation="relu"))
        model.add(tf.keras.layers.BatchNormalization())
    if dropout > 0:
        model.add(tf.keras.layers.Dropout(dropout))
    model.add(tf.keras.layers.Dense(1, activation="linear"))
    model.compile(
        optimizer=tf.keras.optimizers.RMSprop(learning_rate=lr),
        loss="mse", metrics=["mae"]
    )
    return model

def objective_keras(trial):
    layers_key = trial.suggest_categorical("layers", ["64_32", "128_64", "64_32_16"])
    lr         = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    dropout    = trial.suggest_float("dropout", 0.0, 0.3, step=0.1)

    model = build_keras_rmsprop(lr, layers_map[layers_key], dropout)
    model.fit(
        X_train_sc, y_train_log,
        validation_data=(X_test_sc, y_test_log),
        epochs=50, batch_size=32,
        callbacks=[tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=10, restore_best_weights=True, verbose=0
        )],
        verbose=0
    )
    preds = np.expm1(model.predict(X_test_sc, verbose=0).flatten())
    return r2_score(y_test, preds)

study_keras = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42)
)
study_keras.optimize(objective_keras, n_trials=10, show_progress_bar=True)

best_kp = study_keras.best_params
print(f'Лучшие параметры: layers={best_kp["layers"]}  lr={best_kp["lr"]:.6f}  dropout={best_kp["dropout"]}')

ml6 = build_keras_rmsprop(best_kp["lr"], layers_map[best_kp["layers"]], best_kp["dropout"])
history_ml6 = ml6.fit(
    X_train_sc, y_train_log,
    validation_data=(X_test_sc, y_test_log),
    epochs=300, batch_size=32,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=20, restore_best_weights=True, verbose=0
    )],
    verbose=0
)

preds_ml6 = np.expm1(ml6.predict(X_test_sc, verbose=0).flatten())
r2_ml6 = r2_score(y_test, preds_ml6)
results["Keras_rmsprop"] = r2_ml6

ml6.save("./models/keras_rmsprop.keras")
print(f'Keras (rmsprop)  Test R² = {r2_ml6:.4f}')


  0%|          | 0/10 [00:00<?, ?it/s]

c:\Users\Artur\AppData\Local\Programs\Python\Python310\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\Artur\AppData\Local\Programs\Python\Python310\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\Artur\AppData\Local\Programs\Python\Python310\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer

Лучшие параметры: layers=64_32  lr=0.001240  dropout=0.0


c:\Users\Artur\AppData\Local\Programs\Python\Python310\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Keras (rmsprop)  Test R² = 0.8692
